# Forecast Q4 2025 and score it

*Data Analysis in Python · Lesson 10 final exercise*

Your finance team wants to know whether a model trained on the weeks to the end of September could have planned Q4 2025: its Black Friday, its December and its total. Forecast the quarter's weekly revenue, score the forecast against what happened, and write them a readout they can plan 2026 with.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the cell that builds the `sessions` table and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Create the `weekly` table: one row per full Monday-to-Sunday week of the `sessions` table, with the dates as the index, and the columns `sessions`, `transactions` and `revenue`. Add `t`, the week's number from 0, and `week_of_year`, its ISO week number as a whole number.
2. Create the `train` table, the weeks up to `TRAIN_END`, and the `test` table, the weeks after `TRAIN_END` up to `TEST_END`.
3. Create `model`: a regression of the log of revenue on `t` and the week of the year as a category, trained on the `train` table. Then create `forecast`: its forecast of the `test` table's weekly revenue, in dollars, as a Series.
4. Create `mape`: the forecast's MAPE on the `test` table, as a fraction.
5. Create `forecast_total` and `actual_total`: the forecast's and the `test` table's revenue for the whole quarter.
6. Create `forecast_ax`: a line chart of the `test` table's revenue and the forecast, with a legend and a title.
7. Add a text cell for the finance team: how far off the forecast was for the quarter and in a typical week, which weeks it got right and which it missed, and how far they can trust a forecast like it for Q4 2026.
8. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `weekly` table has 104 rows. The `train` table has 91 weeks and the `test` table 13. `mape` is about `0.0615`. `forecast_total` is about $1,130,903, and `actual_total` $1,151,342.92.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
TRAIN_END = "2025-09-28"
TEST_END = "2025-12-28"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly["t"] = range(len(weekly))
weekly["week_of_year"] = weekly.index.isocalendar().week.astype(int)
weekly.shape

In [ ]:
train = weekly[weekly.index <= TRAIN_END]
test = weekly[(weekly.index > TRAIN_END) & (weekly.index <= TEST_END)]
len(train), len(test)

In [ ]:
model = smf.ols("np.log(revenue) ~ t + C(week_of_year)", data=train).fit()
forecast = np.exp(model.predict(test))
mape = ((forecast - test["revenue"]).abs() / test["revenue"]).mean()
forecast_total = forecast.sum()
actual_total = test["revenue"].sum()
mape, forecast_total, actual_total, forecast_total / actual_total - 1

In [ ]:
(forecast / test["revenue"] - 1).round(3)

In [ ]:
forecast_ax = test["revenue"].plot(label="Actual", figsize=(10, 4))
forecast.plot(ax=forecast_ax, label="Forecast")
forecast_ax.legend()
forecast_ax.set_title("Weekly revenue, Q4 2025: forecast from the weeks to September 28 against the actual weeks")
forecast_ax.set_ylabel("Revenue ($)")
plt.show()

**Q4 2025 revenue: what a forecast from September would have said.** The model learned from the 91 weeks to September 28, 2025, and never saw Q4.

- **The quarter.** It forecast $1,130,903, and Q4 brought $1,151,343. That's 1.8% short, about $20,000.
- **A typical week.** It was 6.2% off on average. It got Black Friday week within 1%, $163,344 against $161,833, and it was within 2% for the two weeks around it.
- **What it missed.** For the two weeks before Christmas, the forecast was 16% and 13% short, $32,000 in all. The model only knows December 2024, so it couldn't see that December 2025 would grow faster than the rest of the year. For the week ending November 9, it was 14% too high.
- **For Q4 2026.** You can plan the quarter's total with a forecast like this one, but leave about 5% of room either way. Plan any single week with about 15% of room, and more in December. Update the forecast in early November, once you can see how the quarter is running.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your chart and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.00001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    weekly, train, test = found.get("weekly"), found.get("train"), found.get("test")
    if not isinstance(weekly, pd.DataFrame) or not {"revenue", "t", "week_of_year"} <= set(weekly.columns):
        results.append((False, "weekly should be a DataFrame with one row per full week: sessions, transactions and revenue, plus t and week_of_year."))
    elif len(weekly) == 731:
        results.append((False, "The weekly table has 731 rows: one per day. Total the days by week with resample(\"W\"), as in topic 6.2."))
    elif len(weekly) == 105:
        results.append((False, "The weekly table has 105 rows: the last one is December 29 to 31, 2025, three days. Keep the weeks with 7 days, as in topic 6.2."))
    elif len(weekly) != 104:
        results.append((False, f"The weekly table should have the 104 full weeks of 2024 and 2025. It has {len(weekly):,} rows."))
    elif list(weekly["t"]) != list(range(104)):
        results.append((False, "t should number the weeks from 0 to 103: weekly[\"t\"] = range(len(weekly)), before you split the table."))
    elif not pd.api.types.is_integer_dtype(weekly["week_of_year"]) or str(weekly["week_of_year"].dtype) == "UInt32":
        results.append((False, f"week_of_year is {weekly['week_of_year'].dtype}. Make it a whole number with .astype(int): statsmodels can't read isocalendar()'s UInt32."))
    elif weekly.loc["2025-11-30", "week_of_year"] != 48:
        results.append((False, "week_of_year should be the ISO week number: weekly.index.isocalendar().week.astype(int). Black Friday week 2025, ending November 30, is week 48."))
    elif abs(weekly["revenue"].sum() - 5051311.60) > 0.05:
        results.append((False, f"The weekly table's revenue adds up to ${weekly['revenue'].sum():,.2f}. The 104 full weeks of the clean sessions table add up to $5,051,311.60."))
    elif not isinstance(train, pd.DataFrame) or not isinstance(test, pd.DataFrame):
        results.append((False, "train and test should be DataFrames: the weekly table's rows up to TRAIN_END, and after it up to TEST_END."))
    elif len(test) == 14:
        results.append((False, "The test table has 14 weeks: it starts with the week ending on TRAIN_END, which is also in train. Keep the weeks after TRAIN_END."))
    elif len(train) != 91 or len(test) != 13:
        results.append((False, f"train should have the 91 weeks to September 28, 2025, and test the 13 weeks of Q4 2025. They have {len(train)} and {len(test)}."))
    else:
        results.append((True, "104 full weeks: trained on the 91 to September 28, 2025, and held out the 13 of Q4."))

    forecast = found.get("forecast")
    mape = found.get("mape")
    if not isinstance(forecast, pd.Series) or len(forecast) != 13:
        results.append((False, "forecast should be a Series with the 13 test weeks' revenue: np.exp(model.predict(test))."))
    elif forecast.max() < 100:
        results.append((False, f"forecast's values are around {forecast.mean():.1f}: they're still in logs. Turn them back into dollars with np.exp(model.predict(test))."))
    elif not number(mape):
        results.append((False, f"mape should be a number: ((forecast - test[\"revenue\"]).abs() / test[\"revenue\"]).mean(). It is {mape!r}."))
    elif near(mape, 6.150461, tolerance=0.001):
        results.append((False, "mape is 6.15: in percent. Keep it as a fraction, 0.0615."))
    elif near(mape, 0.031086):
        results.append((False, "mape is 0.0311: too good, because the model trained on the Q4 weeks it's scored on. In October 2025, you wouldn't have had them. Fit the model on the train table only."))
    elif near(mape, 0.299678, 0.302360):
        results.append((False, f"mape is {mape:.4f}: the model has a trend, but no yearly season, so it can't see Black Friday or December coming. Add C(week_of_year) to the formula."))
    elif near(mape, 0.135495):
        results.append((False, "mape is 0.1355: the model isn't in logs, so the season is a fixed number of dollars that doesn't grow with the store, and Q4 comes out too low. Model np.log(revenue), and use np.exp() on the forecast."))
    elif near(mape, 0.311652):
        results.append((False, "mape is 0.3117: that's the same week last year. The brief asks for the model's forecast."))
    elif near(mape, 0.477606):
        results.append((False, "mape is 0.4776: the test weeks' t starts again from 0. Add t to the weekly table before you split it, so the test weeks continue from 91."))
    elif not near(mape, 0.061505):
        results.append((False, f"mape should be the MAPE of np.log(revenue) ~ t + C(week_of_year), trained on train and scored on test: about 0.0615. It is {mape:.4f}."))
    else:
        results.append((True, f"In a typical week of Q4 2025, the forecast was {mape:.1%} off."))

    forecast_total, actual_total = found.get("forecast_total"), found.get("actual_total")
    if not number(forecast_total) or not number(actual_total):
        results.append((False, "forecast_total and actual_total should be numbers: forecast.sum() and test[\"revenue\"].sum()."))
    elif not near(actual_total, 1151342.92, tolerance=0.01):
        results.append((False, f"actual_total should be the test table's revenue, $1,151,342.92. It is ${actual_total:,.2f}."))
    elif not near(forecast_total, 1130902.85, tolerance=0.01):
        results.append((False, f"forecast_total should be the forecast's sum, about $1,130,902.85. It is ${forecast_total:,.2f}."))
    else:
        results.append((True, f"For the quarter, the forecast said ${forecast_total:,.0f} and Q4 brought ${actual_total:,.0f}: {forecast_total / actual_total - 1:+.1%}."))

    ax = found.get("forecast_ax")
    if not isinstance(ax, matplotlib.axes.Axes):
        results.append((False, f"forecast_ax should be the Axes of your line chart: forecast_ax = test[\"revenue\"].plot(label=\"Actual\"). It is a {type(ax).__name__}."))
    else:
        lines = [list(line.get_ydata()) for line in ax.get_lines()]
        actual = [round(value, 2) for value in [53313.3, 63074.03, 66042.37, 68741.49, 63861.56, 59425.99, 78165.33, 124692.62, 161833.46, 134199.0, 118542.74, 107862.08, 51588.95]]
        has_actual = any(set(actual) <= {round(float(value), 2) for value in line} for line in lines)
        has_forecast = isinstance(forecast, pd.Series) and any(len(line) == 13 and all(abs(float(a) - b) < 0.01 for a, b in zip(line, forecast)) for line in lines)
        if not has_actual or not has_forecast:
            results.append((False, f"forecast_ax should have two lines: the test table's revenue and your forecast, 13 weeks each. It has {len(lines)} line{'s' if len(lines) != 1 else ''}{', but not ' + ('the actual weeks' if not has_actual else 'the forecast') if lines else ''}."))
        elif ax.get_legend() is None:
            results.append((False, "The chart has no legend: add label= to each line, then forecast_ax.legend(), so a reader can tell the two lines apart."))
        elif not ax.get_title():
            results.append((False, "The chart has no title. Say what it shows: forecast_ax.set_title(\"...\")."))
        else:
            results.append((True, f"The chart shows the forecast against the actual weeks: \"{ax.get_title()}\"."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the finance team will plan with.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: one row per full week</b></summary>

Topic 6.2.: total by day, then by week, and keep the weeks with 7 days. Then topic 10.2.: number the weeks before you split them, and make the week of the year a whole number, because statsmodels can't read `isocalendar()`'s `UInt32`.

~~~python
daily = sessions.groupby(...)[["sessions", "transactions", "revenue"]].sum()
days_per_week = daily["sessions"].resample(...).count()
weekly = daily.resample(...).sum()
weekly = weekly[days_per_week == ...].copy()
weekly["t"] = range(...)
weekly["week_of_year"] = weekly.index.isocalendar().week.astype(...)
~~~

</details>

<details>
<summary><b>Step 2: the training and holdout weeks</b></summary>

Topic 10.2.: split by date. The week ending on `TRAIN_END` belongs to `train` only.

~~~python
train = weekly[weekly.index <= ...]
test = weekly[(weekly.index > ...) & (weekly.index <= ...)]
len(train), len(test)
~~~

</details>

<details>
<summary><b>Step 3: the model and the forecast</b></summary>

Topic 10.2.: the log of revenue on the trend and the week of the year as a category, fit on `train` only. Turn the forecast back into dollars.

~~~python
model = smf.ols("np.log(...) ~ t + C(...)", data=...).fit()
forecast = np.exp(model.predict(...))
~~~

</details>

<details>
<summary><b>Step 4: the score</b></summary>

Topic 10.2.: each week's absolute error as a share of the actual week, averaged.

~~~python
mape = ((forecast - test[...]).abs() / ...).mean()
~~~

</details>

<details>
<summary><b>Step 5: the quarter</b></summary>

Lesson 2.: a Series' `sum()`.

~~~python
forecast_total = ...
actual_total = ...
forecast_total / actual_total - 1
~~~

</details>

<details>
<summary><b>Step 6: the chart</b></summary>

Topic 10.2.'s video, with lesson 8.'s title: plot the actual weeks, then the forecast on the same Axes.

~~~python
forecast_ax = test[...].plot(label="Actual")
forecast.plot(ax=..., label="Forecast")
forecast_ax.legend()
forecast_ax.set_title(...)
plt.show()
~~~

</details>

<details>
<summary><b>Step 7: the text cell</b></summary>

Choose **Insert → Text cell**. Start with the quarter's total against the actual, then the typical week. Name the weeks the forecast got closest and furthest from, with the numbers: `(forecast / test["revenue"] - 1)` gives you each week's error. End with what the finance team should plan with, and how much room to leave.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly["t"] = range(len(weekly))
weekly["week_of_year"] = weekly.index.isocalendar().week.astype(int)
weekly.shape
```

```python
train = weekly[weekly.index <= TRAIN_END]
test = weekly[(weekly.index > TRAIN_END) & (weekly.index <= TEST_END)]
len(train), len(test)
```

```python
model = smf.ols("np.log(revenue) ~ t + C(week_of_year)", data=train).fit()
forecast = np.exp(model.predict(test))
mape = ((forecast - test["revenue"]).abs() / test["revenue"]).mean()
forecast_total = forecast.sum()
actual_total = test["revenue"].sum()
mape, forecast_total, actual_total, forecast_total / actual_total - 1
```

```python
(forecast / test["revenue"] - 1).round(3)
```

```python
forecast_ax = test["revenue"].plot(label="Actual", figsize=(10, 4))
forecast.plot(ax=forecast_ax, label="Forecast")
forecast_ax.legend()
forecast_ax.set_title("Weekly revenue, Q4 2025: forecast from the weeks to September 28 against the actual weeks")
forecast_ax.set_ylabel("Revenue ($)")
plt.show()
```

```markdown
**Q4 2025 revenue: what a forecast from September would have said.** The model learned from the 91 weeks to September 28, 2025, and never saw Q4.

- **The quarter.** It forecast $1,130,903, and Q4 brought $1,151,343. That's 1.8% short, about $20,000.
- **A typical week.** It was 6.2% off on average. It got Black Friday week within 1%, $163,344 against $161,833, and it was within 2% for the two weeks around it.
- **What it missed.** For the two weeks before Christmas, the forecast was 16% and 13% short, $32,000 in all. The model only knows December 2024, so it couldn't see that December 2025 would grow faster than the rest of the year. For the week ending November 9, it was 14% too high.
- **For Q4 2026.** You can plan the quarter's total with a forecast like this one, but leave about 5% of room either way. Plan any single week with about 15% of room, and more in December. Update the forecast in early November, once you can see how the quarter is running.
```

</details>